In [ ]:
# Importations

import time
from enderscope import SerialUtils, Stage, Panel
import serial
from math import * 
import threading 

In [ ]:
# Variables

rectangle = [20,15] # in mm, x and y

printer_speed = 3 # in mm/s
width_extrusion = 5 # in mm 
number_passages = min(ceil(rectangle[0] /(2 * width_extrusion)),ceil(rectangle[1] /(2 * width_extrusion)))  # computes the smallest integer that is greater than or equal to x.
number_layer = 1 # To modify 
layer_thickness = 2 # in mm

syringe_purge_time = 8 # in seconds, time for the purge
syringe_extrusion_speed = 40 # in mm/s, speed for the purge

horizontal_offset = 12.86 # in mm, horizontal distance between two holes
vertical_offset = 13.6 # in mm, vertical distance between two holes

In [ ]:
# Ports

ports = SerialUtils.serial_ports() # List of ports
print (ports) 

syringe_pump_port = ports[1] # To modify
printer_port = ports[0] # To modify

s = Stage(printer_port, 115200) 
syringe_pump = serial.Serial(port= syringe_pump_port, baudrate=115200, timeout=0.01, writeTimeout=1) # Connexion syringe pump

In [ ]:
# HOMING X Y
s.write_code("G28 X Y") # Home the printer

In [ ]:
# Set absolute positioning mode
s.set_absolute()

In [ ]:
# Set relative positioning mode
s.set_relative()

In [ ]:
s.write_code("G0 X100 Y100") # go to x100y100

In [ ]:
s.get_position() # Get current position

In [ ]:
s.write_code(f"M203 X{30} Y{30}") # Set max speed for x and y

In [ ]:
s.write_code("G0 Z10")

In [ ]:
s.write_code("G92 Z0") # set z0

In [ ]:
s.write_code("M208 X0:200 Y0:200 Z0:200") # set the limits of the printer (marche pas)

In [ ]:
# ============================================================
# FUNCTIONS
# ============================================================

# --- Syringe pump : messages ---

def start_extrusion(letter_syringe_pump): 
    return(f"{letter_syringe_pump}\n".encode('utf8')) 

def stop_extrusion():
    return b"S\n"

def set_extrusion_speed(speed): # in steps per second
    return(f"V{speed}\n".encode('utf8'))


# --- Emergency stop : stops all stage and syringe pump movements ---
# Call this from a KeyboardInterrupt handler (e.g. when the cell execution is killed)
# to halt any ongoing movement as fast as possible.

def emergency_stop():
    try:
        syringe_pump.write(stop_extrusion())  # stop the syringe pump extrusion
    except Exception as e:
        print(f"Error while stopping the syringe pump: {e}")

    try:
        s.serial.write(b"M410\n")  # Quickstop: halts stage motion immediately
    except Exception as e:
        print(f"Error while stopping the stage: {e}")

    print("Emergency stop: stage and syringe pump have been stopped.")


# --- Purge ---

def purge_syringe(letter_syringe_pump, stop_event):
    syringe_pump.write(start_extrusion(letter_syringe_pump))
    stop_event.wait(syringe_purge_time)  # interruptible wait for the purge duration
    syringe_pump.write(stop_extrusion())
    time.sleep(1)

def purge():
    stop_event = threading.Event()
    purge_A = threading.Thread(target=purge_syringe, args=('A', stop_event), daemon=True)
    purge_A.start()
    try:
        purge_A.join()
    except KeyboardInterrupt:
        stop_event.set()  # release the waiting thread so it stops the extrusion right away
        emergency_stop()
        purge_A.join()
        raise


# --- Rectangle ---

def function_rectangle(): 

    x_rectangle = rectangle[0]
    y_rectangle = rectangle[1]

    for i in range (number_passages):
        s.write_code(f"M203 X{printer_speed}")
        s.write_code(f"M203 Y{printer_speed}")
        s.move_axis('x', x_rectangle)
        s.move_axis ('y', y_rectangle)
        s.move_axis('x', -x_rectangle)
        s.move_axis('y', - (y_rectangle - width_extrusion))

        s.move_axis('x', width_extrusion)
        
        x_rectangle = x_rectangle - (2 * width_extrusion) # The next inner rectangle will have two fewer layers
        y_rectangle = y_rectangle - (2*width_extrusion)

    s.write_code(f"M400")


# --- Go to the position and adjust according to the syringe pump ---

def go(letter_syringe_pump):
    s.write_code(f"M203 X{printer_speed}")
    s.write_code(f"M203 Y{printer_speed}")
    if letter_syringe_pump == 'A':
        pass

    if letter_syringe_pump =='B':
        s.move_axis('x', rectangle[0] - horizontal_offset)   # a horizontal distance equal to 'horizontal_offset' between A and B
        s.move_axis('z', - (number_layer - 1) * layer_thickness) # Return to the initial position in z 
        s.write_code(f"M400")

    if letter_syringe_pump == 'C':
        s.move_axis('x', rectangle[0] + horizontal_offset) 
        s.move_axis('y', - vertical_offset) # a vertical distance equal to 'vertical_offset' betwween B and C 
        s.move_axis('z', - (number_layer - 1) * layer_thickness) # Return to the initial position in z 
        s.write_code(f"M400")


# --- Go to the initial position ---

def go_initial_position():
    s.write_code(f"M203 X{printer_speed}")
    s.write_code(f"M203 Y{printer_speed}")
    s.move_axis('x', - width_extrusion * number_passages)  # Return to the initial position in x
    s.move_axis('y',- width_extrusion * number_passages)   # Return to the initial position in y
    s.write_code(f"M400")


# --- Print rectangle ---

def print_rectangle(letter_syringe_pump):
    go(letter_syringe_pump)

    try:
        for i in range(number_layer): # Repeat for each layer 
            if i > 1:
                s.move_axis('z', layer_thickness) # With each new layer, the height is increased 
            syringe_pump.write(set_extrusion_speed(syringe_extrusion_speed))
            syringe_pump.write(start_extrusion(letter_syringe_pump)) 
            function_rectangle()
            syringe_pump.write(stop_extrusion())
            s.move_axis('z', 30) # Move up for clearance
            go_initial_position() 
    except KeyboardInterrupt:
        emergency_stop()  # stop the stage and syringe pump if execution is killed
        raise


In [ ]:
syringe_pump.write(set_extrusion_speed(200)) # Set the speed of the syringe pump

In [ ]:
# Choose the purge point FOR A (at the bottom left)
#      ________
#   C | °    ° | D
#   A | °    ° | B  
#      ¯¯¯¯¯¯¯¯
p = Panel(s) 

In [ ]:
# Purge 
purge()

In [ ]:
# Choose the printing location
p = Panel(s) 

In [ ]:
# Print the rectangles
print_rectangle('A')
#print_rectangle('B')
#print_rectangle('C')

In [ ]:
go_initial_position() # Return to the initial position

In [ ]:
# Control panel (syringe_printer.py)
# Standalone alternative to the cells above: pick ports and click "Connect",
# then use the Home / Purge / Print A-B-C / STOP buttons.

from syringe_printer import SyringePrinterController, PrintControlPanel

controller = SyringePrinterController()
control_panel = PrintControlPanel(controller)
